> **Course website copy · Team 3.** This is the team's final submission as handed in, with their run outputs. Names are removed. The one cell below was added for the website: it fetches the team's data files from the course repository, so *Runtime → Run all* works without uploading anything. Everything after it is the team's own work.

In [ ]:
# Added for the course website (not part of the team's submission):
# copy the team's data files from the course repository into the folder the notebook expects.
import os, shutil, subprocess
SRC = "week07/projects/team3-smartphone/data"
TARGET = "/content/team3_data"
if not os.path.isdir("course_copy"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/TH-Koln-Bartnik/sscm-2026-student", "course_copy"], check=True)
    subprocess.run(["git", "-C", "course_copy", "sparse-checkout", "set", SRC], check=True)
os.makedirs(TARGET, exist_ok=True)
for name in os.listdir(os.path.join("course_copy", SRC)):
    shutil.copy(os.path.join("course_copy", SRC, name), os.path.join(TARGET, name))
print(len(os.listdir(TARGET)), "data files in", os.path.abspath(TARGET))


# Group project · the check cell

The **check cell** is a fixed piece of code that tests your dataset and your baseline results with five tests (T1–T5 in the project brief). Think of it as the technical inspection for a car: it does not say whether your car is beautiful, only whether it is safe to drive. It prints PASS or FAIL per test and, below each line, the numbers it used, so a FAIL tells you where to look.

This notebook shows the check cell working on the UrbanRide bike data. Run it top to bottom once (Runtime → Run all), then copy **the last cell** into your own project notebook, unchanged.

**What the check cell needs from you:** two names, defined before it runs.

| Name | What it is |
|---|---|
| `DATA_DIR` | the folder that holds your ten CSV files |
| `baseline` | your baseline results, as a list of 10 dictionaries (one per seed 1–10) or one dictionary with the means |

## Step 1 · Data: the bike files as a stand-in for yours

This cell copies the course repository into Colab and points `DATA_DIR` at the bike data. The bike folder has no `sources.csv`, so the cell writes a two-row example; yours has one row per number you entered. In your own notebook, `DATA_DIR` is the folder where you uploaded your files.

In [15]:
# 1. Load the project data from GitHub
import os
import heapq
import numpy as np
import pandas as pd
from IPython.display import display

# Course website copy: the data come from the cell above (the team cloned them from their own GitHub repository).
DATA_DIR = "/content/team3_data"

required_files = [
    "products.csv", "bom.csv", "workcenters.csv", "routing.csv",
    "process_environment.csv", "suppliers.csv", "locations.csv",
    "network_lanes.csv", "freight_rates.csv", "sources.csv"
]
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing project files in DATA_DIR: {missing}")

products = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
bom = pd.read_csv(os.path.join(DATA_DIR, "bom.csv"))
workcenters = pd.read_csv(os.path.join(DATA_DIR, "workcenters.csv"))
routing = pd.read_csv(os.path.join(DATA_DIR, "routing.csv"))

PRODUCT_ID = "PH-FLAG"
product_bom = bom[bom["product_id"] == PRODUCT_ID].copy()
product_routing = routing[routing["product_id"] == PRODUCT_ID].sort_values("seq").copy()

if product_bom.empty or product_routing.empty:
    raise ValueError(f"No BOM or routing data found for product {PRODUCT_ID}.")

print("Product:", products.loc[products.product_id == PRODUCT_ID, "product_name"].iloc[0])
print("Production route:", " → ".join(
    workcenters.set_index("workcenter_id").loc[
        product_routing.workcenter_id, "workcenter_name"
    ].tolist()
))
print("Material cost per started phone:", round(product_bom["unit_cost_eur"].sum(), 2), "EUR")


Product: Samsung Galaxy S25 Ultra 256GB
Production route: SMT → Board test → Assembly → Final test → Packaging
Material cost per started phone: 390 EUR


## Step 2 · Baseline results: the eight numbers your model must report

Your model notebook produces these numbers in the baseline scenario. The values below are the means of ten seeds of a simple bike-line model; yours come from your own model.

| Key | Meaning | How to measure it in your model |
|---|---|---|
| `weeks_simulated` | weeks in the run | 52 |
| `hours_simulated` | length of the run in clock hours | e.g. 52 × 5 days × 8 h = 2,080 if your clock counts working hours only |
| `units_started` | units that entered the line | count at the first station |
| `units_finished` | units that left the last station | count at the last station |
| `key_components_used` | key components taken from stock into the line | count where the component enters (`added_at_workcenter`) |
| `avg_units_on_line` | average number of units between start and finish, over the whole run | see the helper below |
| `avg_hours_on_line` | average hours from start to finish of a finished unit | finish time − start time, averaged |
| `cost_per_unit_eur` | one-year total cost ÷ units finished | the total-cost formula of the brief |

**Use one clock for everything.** If `hours_simulated` counts working hours only, `avg_hours_on_line` must count working hours too. Little's Law (T3) compares the two, and a mixed clock is the most common reason for a FAIL.
**Synchronization note:** The material-supply simulation and the final animation use the same stochastic supplier-production and transport lead-time logic. Run the notebook cells in order and use the same seed (`1`) when comparing results.


In [16]:
# 2. Helper: calculate the average WIP and average flow time on one working-hour clock
def line_averages(start_h, finish_h, hours_simulated):
    start_h = np.asarray(start_h, dtype=float)
    finish_h = np.asarray(finish_h, dtype=float)
    completed = finish_h <= hours_simulated

    # Units still unfinished at the end remain on the line until the end of the run.
    clipped_finish = np.minimum(finish_h, hours_simulated)
    avg_units_on_line = np.maximum(0, clipped_finish - start_h).sum() / hours_simulated

    avg_hours_on_line = (
        (finish_h[completed] - start_h[completed]).mean()
        if completed.any() else 0.0
    )
    return float(avg_units_on_line), float(avg_hours_on_line)


In [17]:
# 3. Define and run one 52-week baseline simulation
WEEKS = 52
HOURS_SIMULATED = 52 * 5 * 8  # working hours only
TARGET_UNITS_PER_WEEK = 700
UNITS_TO_START = TARGET_UNITS_PER_WEEK * WEEKS

def simulate_baseline(seed):
    rng = np.random.default_rng(seed)

    # Release 700 phones evenly over each year's 2,080 working hours.
    starts = np.linspace(
        0,
        HOURS_SIMULATED - (40 / TARGET_UNITS_PER_WEEK),
        UNITS_TO_START
    )
    station_arrivals = starts.copy()

    route = product_routing.merge(
        workcenters[[
            "workcenter_id", "num_servers", "availability", "process_time_cv"
        ]],
        on="workcenter_id",
        how="left",
        validate="one_to_one"
    )

    for _, station in route.iterrows():
        servers = int(station["num_servers"])
        availability = float(station["availability"])
        cv = float(station["process_time_cv"])
        mean_process_hours = float(station["process_time_min"]) / 60.0

        if servers < 1 or not (0 < availability <= 1):
            raise ValueError(f"Invalid capacity settings at {station['workcenter_id']}.")

        # Effective processing time includes the unavailable share of the workcentre.
        effective_mean = mean_process_hours / availability

        # Lognormal service-time distribution with the specified mean and CV.
        sigma_squared = np.log(1 + cv**2)
        sigma = np.sqrt(sigma_squared)
        mu = np.log(effective_mean) - sigma_squared / 2
        service_times = rng.lognormal(mu, sigma, len(station_arrivals))

        # FCFS queue with the configured number of parallel servers.
        server_free_times = [0.0] * servers
        heapq.heapify(server_free_times)
        station_departures = np.empty(len(station_arrivals))

        for i, arrival in enumerate(station_arrivals):
            earliest_server = heapq.heappop(server_free_times)
            service_start = max(float(arrival), earliest_server)
            departure = service_start + float(service_times[i])
            station_departures[i] = departure
            heapq.heappush(server_free_times, departure)

        station_arrivals = station_departures

    finish_times = station_arrivals
    units_finished = int(np.sum(finish_times <= HOURS_SIMULATED))
    avg_units, avg_hours = line_averages(starts, finish_times, HOURS_SIMULATED)

    # The key component is the component in the BOM that is explicitly assigned
    # to the first workcentre (added_at_workcenter).
    key_rows = product_bom[
        (product_bom["component_id"] == "mem") &
        (product_bom["added_at_workcenter"] == product_routing.iloc[0]["workcenter_id"])
    ]
    if len(key_rows) != 1:
        raise ValueError("Expected exactly one 'mem' component added at the first workcentre.")
    memory_qty = float(key_rows.iloc[0]["qty_per_unit"])
    key_components_used = UNITS_TO_START * memory_qty

    # Baseline check-cell cost: annual material spend for started units plus
    # annual workcentre fixed costs, divided by units completed.
    # This is the baseline check metric, not the full scenario cost model.
    material_cost_per_phone = float(product_bom["unit_cost_eur"].sum())
    annual_material_cost = material_cost_per_phone * UNITS_TO_START
    annual_fixed_cost = float(workcenters["fixed_cost_eur_week"].sum()) * WEEKS
    cost_per_unit = (
        (annual_material_cost + annual_fixed_cost) / units_finished
        if units_finished > 0 else float("inf")
    )

    return {
        "weeks_simulated": WEEKS,
        "hours_simulated": HOURS_SIMULATED,
        "units_started": int(UNITS_TO_START),
        "units_finished": units_finished,
        "key_components_used": int(key_components_used),
        "avg_units_on_line": round(avg_units, 4),
        "avg_hours_on_line": round(avg_hours, 6),
        "cost_per_unit_eur": round(cost_per_unit, 4),
    }

print("Simulation function defined.")


Simulation function defined.


In [18]:
# 4. Run the baseline ten times (seeds 1–10)
baseline = [simulate_baseline(seed) for seed in range(1, 11)]
baseline_df = pd.DataFrame(baseline)

print("Results for the 10 baseline runs:")
display(baseline_df)

baseline_mean = {
    key: float(baseline_df[key].mean())
    for key in baseline_df.columns
}
print("\nMean baseline values:")
for key, value in baseline_mean.items():
    print(f"{key}: {value:.4f}")


Results for the 10 baseline runs:


,weeks_simulated,hours_simulated,units_started,units_finished,key_components_used,avg_units_on_line,avg_hours_on_line,cost_per_unit_eur
0,52,2080,36400,36397,36400,2.8398,0.162279,397.4613
1,52,2080,36400,36398,36400,2.8483,0.162762,397.4504
2,52,2080,36400,36399,36400,2.8456,0.162607,397.4395
3,52,2080,36400,36398,36400,2.8480,0.162748,397.4504
4,52,2080,36400,36397,36400,2.8545,0.163117,397.4613
5,52,2080,36400,36398,36400,2.8448,0.162565,397.4504
6,52,2080,36400,36398,36400,2.8356,0.162041,397.4504
7,52,2080,36400,36398,36400,2.8443,0.162535,397.4504
8,52,2080,36400,36396,36400,2.8520,0.162971,397.4722
9,52,2080,36400,36398,36400,2.8534,0.163054,397.4504



Mean baseline values:
weeks_simulated: 52.0000
hours_simulated: 2080.0000
units_started: 36400.0000
units_finished: 36397.7000
key_components_used: 36400.0000
avg_units_on_line: 2.8466
avg_hours_on_line: 0.1627
cost_per_unit_eur: 397.4537


In [19]:
# 5. Capacity check before the official course check
capacity_check = product_routing.merge(
    workcenters,
    on="workcenter_id",
    how="left",
    validate="one_to_one"
)
capacity_check["utilization_share"] = (
    UNITS_TO_START * capacity_check["process_time_min"]
    / (
        capacity_check["num_servers"]
        * capacity_check["shift_hours"]
        * 60
        * capacity_check["days_per_week"]
        * capacity_check["availability"]
        * WEEKS
    )
)
display(capacity_check[[
    "workcenter_id", "workcenter_name", "utilization_share"
]])

if (capacity_check["utilization_share"] >= 1).any():
    raise ValueError("Capacity check failed: at least one workcentre is at or above 100% utilization.")
print("Capacity check: PASS")


,workcenter_id,workcenter_name,utilization_share
0,WC1,SMT,0.648148
1,WC2,Board test,0.324074
2,WC3,Assembly,0.514706
3,WC4,Final test,0.486111
4,WC5,Packaging,0.153509


Capacity check: PASS


In [20]:

# 6. Define the three supply chain scenarios

SCENARIOS = {
    "Baseline": {
        "description": "Normal operation without supply chain disruption.",
        "disruption_weeks": 0,
        "extra_lead_time_days": 0,
        "freight_cost_increase": 0.0,
        "dual_sourcing": False,
        "poland_share": 0.0
    },

    "Red Sea Crisis": {
        "description": "Asian shipments are rerouted around the Cape of Good Hope.",
        "disruption_weeks": 12,
        "extra_lead_time_days": 10,
        "freight_cost_increase": 0.20,
        "dual_sourcing": False,
        "poland_share": 0.0
    },

    "Red Sea Crisis + Dual Sourcing": {
        "description": "A second supplier in Poland is activated.",
        "disruption_weeks": 12,
        "extra_lead_time_days": 10,
        "freight_cost_increase": 0.20,
        "dual_sourcing": True,
        "poland_share": 0.50
    }
}

print("Three scenarios defined:")
for name, scenario in SCENARIOS.items():
    print("-", name, ":", scenario["description"])


Three scenarios defined:
- Baseline : Normal operation without supply chain disruption.
- Red Sea Crisis : Asian shipments are rerouted around the Cape of Good Hope.
- Red Sea Crisis + Dual Sourcing : A second supplier in Poland is activated.


In [21]:
# 7. Load supplier and transport data for the scenarios

suppliers = pd.read_csv(os.path.join(DATA_DIR, "suppliers.csv"))
network_lanes = pd.read_csv(os.path.join(DATA_DIR, "network_lanes.csv"))
freight_rates = pd.read_csv(os.path.join(DATA_DIR, "freight_rates.csv"))
process_environment = pd.read_csv(
    os.path.join(DATA_DIR, "process_environment.csv")
)

print("KEY COMPONENT IN THE BOM")
display(product_bom[product_bom["component_id"] == "mem"])

print("AVAILABLE SUPPLIERS")
display(suppliers[suppliers["component_id"] == "mem"])

print("TRANSPORT ROUTES")
display(network_lanes)

print("FREIGHT RATES")
display(freight_rates)


KEY COMPONENT IN THE BOM


,product_id,component_id,component_name,qty_per_unit,mass_kg,unit_cost_eur,supplier_id,added_at_workcenter
0,PH-FLAG,mem,memory module (12GB+256GB),1,0.005,70,S1,WC1


AVAILABLE SUPPLIERS


,supplier_id,supplier_name,country,component_id,unit_price_eur,promised_lead_days,lead_time_mean_days,lead_time_sd_days,capacity_per_month,min_per_month,kg_co2e_per_unit,location_id
0,S1,MemCore (rep. major DRAM/NAND maker),KR,mem,70,15,35,4,60000,20000,0.8,LOC-S1
1,S2,MemAsia (rep. second maker),TW,mem,72,12,48,3,50000,10000,0.7,LOC-S2
2,S3,MemEU (rep. nearer maker),PL,mem,75,5,7,1,30000,5000,0.4,LOC-S3


TRANSPORT ROUTES


,lane_id,origin_id,destination_id,mode_id,distance_km,transit_days_mean,transit_days_sd
0,L1-SEA,LOC-S1,LOC-PORT,sea,19000,30,4
1,L2-SEA,LOC-S2,LOC-PORT,sea,19003,43,5
2,L3-TRUCK,LOC-PORT,LOC-PLANT,truck,500,2,1
3,L4-TRUCK,LOC-S3,LOC-PLANT,truck,1100,3,1


FREIGHT RATES


,lane_id,mode_id,rate_eur_per_shipment,rate_eur_per_tkm,min_charge_eur
0,L1-SEA,sea,1500,0.08,2000
1,L2-SEA,sea,1500,0.08,2000
2,L3-TRUCK,truck,300,0.35,400
3,L4-TRUCK,truck,300,0.35,400


In [22]:
# 8. Simulate memory-chip supply for all three scenarios
# IMPORTANT: This function intentionally uses the same stochastic lead-time model
# as the final animation, so both outputs are based on identical weekly supply data.

WORKING_HOURS_PER_WEEK = 40
WEEKLY_MEMORY_DEMAND = TARGET_UNITS_PER_WEEK
INITIAL_MEMORY_STOCK = 5 * WEEKLY_MEMORY_DEMAND


def simulate_material_supply(scenario_name, seed=1):
    rng = np.random.default_rng(seed)
    scenario = SCENARIOS[scenario_name]

    stock = INITIAL_MEMORY_STOCK
    deliveries = {}  # week -> arrived quantity

    released_units = []
    stopped_hours = 0.0
    weekly_results = []

    s1 = suppliers.loc[suppliers["supplier_id"] == "S1"].iloc[0]
    s3 = suppliers.loc[suppliers["supplier_id"] == "S3"].iloc[0]

    l1_sea = network_lanes.loc[network_lanes["lane_id"] == "L1-SEA"].iloc[0]
    l3_truck = network_lanes.loc[network_lanes["lane_id"] == "L3-TRUCK"].iloc[0]
    l4_truck = network_lanes.loc[network_lanes["lane_id"] == "L4-TRUCK"].iloc[0]

    for week in range(WEEKS):
        # Receive all deliveries scheduled for this week.
        stock += deliveries.get(week, 0)

        # Determine the sourcing allocation.
        if scenario["dual_sourcing"]:
            s1_share = 1.0 - scenario["poland_share"]
            s3_share = scenario["poland_share"]
        else:
            s1_share = 1.0
            s3_share = 0.0

        s1_order = WEEKLY_MEMORY_DEMAND * s1_share
        s3_order = WEEKLY_MEMORY_DEMAND * s3_share

        # Asian supplier: production lead time + sea freight + truck transport.
        if s1_order > 0:
            production_lead = rng.normal(
                float(s1["lead_time_mean_days"]),
                float(s1["lead_time_sd_days"])
            )
            production_lead = max(1.0, production_lead)

            if week < scenario["disruption_weeks"]:
                production_lead += scenario["extra_lead_time_days"]

            sea_days = rng.normal(
                float(l1_sea["transit_days_mean"]),
                float(l1_sea["transit_days_sd"])
            )
            sea_days = max(1.0, sea_days)

            truck_days = rng.normal(
                float(l3_truck["transit_days_mean"]),
                float(l3_truck["transit_days_sd"])
            )
            truck_days = max(0.5, truck_days)

            total_lead_days = production_lead + sea_days + truck_days
            arrival_week = week + int(np.ceil(total_lead_days / 7))

            if arrival_week < WEEKS:
                deliveries[arrival_week] = (
                    deliveries.get(arrival_week, 0) + s1_order
                )

        # Polish supplier: production lead time + truck transport.
        if s3_order > 0:
            production_lead = rng.normal(
                float(s3["lead_time_mean_days"]),
                float(s3["lead_time_sd_days"])
            )
            production_lead = max(1.0, production_lead)

            truck_days = rng.normal(
                float(l4_truck["transit_days_mean"]),
                float(l4_truck["transit_days_sd"])
            )
            truck_days = max(0.5, truck_days)

            total_lead_days = production_lead + truck_days
            arrival_week = week + int(np.ceil(total_lead_days / 7))

            if arrival_week < WEEKS:
                deliveries[arrival_week] = (
                    deliveries.get(arrival_week, 0) + s3_order
                )

        # Release only the number of phones for which memory chips are available.
        available_units = min(WEEKLY_MEMORY_DEMAND, int(stock))
        stock -= available_units

        missing_units = WEEKLY_MEMORY_DEMAND - available_units
        weekly_stopped_hours = (
            missing_units / WEEKLY_MEMORY_DEMAND
        ) * WORKING_HOURS_PER_WEEK
        stopped_hours += weekly_stopped_hours

        # Distribute production starts evenly over the working week.
        if available_units > 0:
            week_starts = (
                week * WORKING_HOURS_PER_WEEK
                + np.linspace(
                    0,
                    WORKING_HOURS_PER_WEEK,
                    available_units,
                    endpoint=False
                )
            )
            released_units.extend(week_starts.tolist())

        weekly_results.append({
            "week": week + 1,
            "memory_stock": stock,
            "units_released": available_units,
            "missing_units": missing_units,
            "stopped_hours": weekly_stopped_hours
        })

    return {
        "starts": np.array(released_units, dtype=float),
        "stopped_hours": stopped_hours,
        "weekly_results": pd.DataFrame(weekly_results)
    }


# Test the material supply model with all three scenarios.
# Use the same seed as the final animation for directly comparable results.
for scenario_name in SCENARIOS:
    result = simulate_material_supply(scenario_name, seed=1)

    print("\n", scenario_name)
    print("Units released:", len(result["starts"]))
    print("Hours stopped:", round(result["stopped_hours"], 2))



 Baseline
Units released: 32200
Hours stopped: 240.0

 Red Sea Crisis
Units released: 31500
Hours stopped: 280.0

 Red Sea Crisis + Dual Sourcing
Units released: 35000
Hours stopped: 80.0


In [23]:

# 9. Simulate production for all three supply chain scenarios

def simulate_scenario_production(scenario_name, seed=1):
    # Get material supply results from cell 8 (synchronized with the animation)
    supply = simulate_material_supply(scenario_name, seed)

    starts = supply["starts"]
    stopped_hours = supply["stopped_hours"]

    rng = np.random.default_rng(seed)

    # Prepare production route
    route = product_routing.merge(
        workcenters[
            ["workcenter_id", "num_servers", "availability", "process_time_cv"]
        ],
        on="workcenter_id",
        how="left",
        validate="one_to_one"
    )

    station_arrivals = starts.copy()

    # Simulate each production workcentre
    for _, station in route.iterrows():
        servers = int(station["num_servers"])
        availability = float(station["availability"])
        cv = float(station["process_time_cv"])
        mean_process_hours = float(station["process_time_min"]) / 60

        effective_mean = mean_process_hours / availability

        sigma_squared = np.log(1 + cv**2)
        sigma = np.sqrt(sigma_squared)
        mu = np.log(effective_mean) - sigma_squared / 2

        service_times = rng.lognormal(
            mu, sigma, len(station_arrivals)
        )

        server_free_times = [0.0] * servers
        heapq.heapify(server_free_times)

        departures = np.empty(len(station_arrivals))

        for i, arrival in enumerate(station_arrivals):
            earliest_server = heapq.heappop(server_free_times)

            service_start = max(float(arrival), earliest_server)
            departure = service_start + float(service_times[i])

            departures[i] = departure
            heapq.heappush(server_free_times, departure)

        station_arrivals = departures

    # Count completed phones within the simulation period
    units_finished = int(
        np.sum(station_arrivals <= HOURS_SIMULATED)
    )

    # Calculate average work in progress and flow time
    avg_units, avg_hours = line_averages(
        starts,
        station_arrivals,
        HOURS_SIMULATED
    )

    return {
        "scenario": scenario_name,
        "units_started": len(starts),
        "units_finished": units_finished,
        "hours_stopped": round(stopped_hours, 2),
        "avg_units_on_line": round(avg_units, 2),
        "avg_hours_on_line": round(avg_hours, 2)
    }


# Run all three scenarios
scenario_production_results = []

for scenario_name in SCENARIOS:
    result = simulate_scenario_production(scenario_name, seed=1)
    scenario_production_results.append(result)

scenario_production_df = pd.DataFrame(scenario_production_results)

print("PRODUCTION SIMULATION RESULTS")
display(scenario_production_df)


PRODUCTION SIMULATION RESULTS


,scenario,units_started,units_finished,hours_stopped,avg_units_on_line,avg_hours_on_line
0,Baseline,32200,32196,240.0,2.52,0.16
1,Red Sea Crisis,31500,31498,280.0,2.46,0.16
2,Red Sea Crisis + Dual Sourcing,35000,34998,80.0,2.73,0.16


In [24]:

# 10. Calculate costs and CO2 emissions for all scenarios

# Assumptions
ELECTRICITY_CO2_FACTOR = 0.344  # kg CO2 per kWh
SHIPMENT_SIZE = TARGET_UNITS_PER_WEEK
ANNUAL_INVENTORY_HOLDING_RATE = 0.20

# Identify suppliers
supplier_s1 = suppliers.loc[
    suppliers["supplier_id"] == "S1"
].iloc[0]

supplier_s3 = suppliers.loc[
    suppliers["supplier_id"] == "S3"
].iloc[0]

# Identify memory component
memory_row = product_bom.loc[
    product_bom["component_id"] == "mem"
].iloc[0]

memory_qty = float(memory_row["qty_per_unit"])
memory_mass_kg = float(memory_row["mass_kg"]) * memory_qty

# Material costs excluding memory
other_material_cost = float(
    product_bom.loc[
        product_bom["component_id"] != "mem",
        "unit_cost_eur"
    ].sum()
)

# Calculate freight costs
def calculate_lane_freight(lane_id, shipment_mass_kg):

    lane = network_lanes.loc[
        network_lanes["lane_id"] == lane_id
    ].iloc[0]

    rate = freight_rates.loc[
        freight_rates["lane_id"] == lane_id
    ].iloc[0]

    tonne_km = (
        float(lane["distance_km"]) *
        shipment_mass_kg / 1000
    )

    variable_cost = (
        float(rate["rate_eur_per_tkm"]) * tonne_km
    )

    return max(
        float(rate["min_charge_eur"]),
        float(rate["rate_eur_per_shipment"]) + variable_cost
    )


s1_freight_per_phone = (
    calculate_lane_freight(
        "L1-SEA", memory_mass_kg * SHIPMENT_SIZE
    )
    + calculate_lane_freight(
        "L3-TRUCK", memory_mass_kg * SHIPMENT_SIZE
    )
) / SHIPMENT_SIZE

s3_freight_per_phone = (
    calculate_lane_freight(
        "L4-TRUCK", memory_mass_kg * SHIPMENT_SIZE
    )
) / SHIPMENT_SIZE


# Calculate electricity consumption per phone
energy_kwh_per_phone = 0.0

for _, station in product_routing.iterrows():

    wc_id = station["workcenter_id"]

    environment = process_environment.loc[
        process_environment["workcenter_id"] == wc_id
    ].iloc[0]

    process_hours = float(station["process_time_min"]) / 60

    availability = float(
        workcenters.loc[
            workcenters["workcenter_id"] == wc_id,
            "availability"
        ].iloc[0]
    )

    energy_kwh_per_phone += (
        float(environment["kw_running"])
        * process_hours / availability
    )


# Calculate costs and emissions for each scenario
cost_co2_results = []

for _, production in scenario_production_df.iterrows():

    scenario_name = production["scenario"]
    scenario = SCENARIOS[scenario_name]

    # Supplier shares
    poland_share = scenario["poland_share"]
    asia_share = 1 - poland_share

    # Material costs
    memory_cost = memory_qty * (
        asia_share * float(supplier_s1["unit_price_eur"])
        + poland_share * float(supplier_s3["unit_price_eur"])
    )

    material_cost = other_material_cost + memory_cost

    # Freight costs
    freight_cost = (
        asia_share * s1_freight_per_phone
        + poland_share * s3_freight_per_phone
    )

    if scenario["disruption_weeks"] > 0:
        freight_cost = (
            asia_share * s1_freight_per_phone
            * (1 + scenario["freight_cost_increase"])
            + poland_share * s3_freight_per_phone
        )

    # Production costs
    annual_fixed_cost = (
        float(workcenters["fixed_cost_eur_week"].sum()) * WEEKS
    )

    units_finished = int(production["units_finished"])

    production_cost = (
        annual_fixed_cost / units_finished
        if units_finished > 0 else 0
    )

    # Inventory holding costs
    supply_result = simulate_material_supply(
        scenario_name, seed=1
    )

    average_memory_stock = float(
        supply_result["weekly_results"]["memory_stock"].mean()
    )

    average_memory_price = (
        asia_share * float(supplier_s1["unit_price_eur"])
        + poland_share * float(supplier_s3["unit_price_eur"])
    )

    average_inventory_value = (
        average_memory_stock * average_memory_price
    )

    annual_holding_cost = (
        average_inventory_value * ANNUAL_INVENTORY_HOLDING_RATE
    )

    holding_cost_per_phone = (
        annual_holding_cost / units_finished
        if units_finished > 0 else 0
    )

    # Total cost per phone
    total_cost = (
        material_cost
        + freight_cost
        + production_cost
        + holding_cost_per_phone
    )

    # CO2 emissions
    supplier_emissions = memory_qty * (
        asia_share * float(supplier_s1["kg_co2e_per_unit"])
        + poland_share * float(supplier_s3["kg_co2e_per_unit"])
    )

    electricity_emissions = (
        energy_kwh_per_phone * ELECTRICITY_CO2_FACTOR
    )

    total_emissions = (
        supplier_emissions + electricity_emissions
    )

    cost_co2_results.append({
        "scenario": scenario_name,
        "material_cost_eur": round(material_cost, 2),
        "freight_cost_eur": round(freight_cost, 2),
        "production_cost_eur": round(production_cost, 2),
        "holding_cost_eur": round(holding_cost_per_phone, 2),
        "total_cost_per_unit_eur": round(total_cost, 2),
        "kg_co2_per_unit": round(total_emissions, 4)
    })


cost_co2_df = pd.DataFrame(cost_co2_results)

print("COST AND CO2 RESULTS")
display(cost_co2_df)


COST AND CO2 RESULTS


,scenario,material_cost_eur,freight_cost_eur,production_cost_eur,holding_cost_eur,total_cost_per_unit_eur,kg_co2_per_unit
0,Baseline,390.0,3.43,8.40,0.30,402.13,0.8904
1,Red Sea Crisis,390.0,4.11,8.58,0.44,403.14,0.8904
2,Red Sea Crisis + Dual Sourcing,392.5,2.34,7.73,0.26,402.83,0.6904


### Optional helper: average units on the line from start and finish times

If your model records, for every unit, the hour it started and the hour it finished, this function returns the two averages Little's Law needs. A unit that has not finished by the end of the run counts as "on the line" until the end.

In [25]:
import numpy as np

def line_averages(start_h, finish_h, hours_simulated):
    """start_h, finish_h: one entry per unit started (use np.inf for unfinished units)."""
    start_h, finish_h = np.asarray(start_h, float), np.asarray(finish_h, float)
    done = finish_h <= hours_simulated
    avg_units_on_line = (np.minimum(finish_h, hours_simulated) - start_h).sum() / hours_simulated
    avg_hours_on_line = (finish_h[done] - start_h[done]).mean()
    return round(float(avg_units_on_line), 2), round(float(avg_hours_on_line), 2)

# tiny example: 3 units on a 10-hour clock, the third one still on the line at the end
print(line_averages([0, 2, 8], [3, 6, np.inf], 10))

(0.9, 3.5)


## Step 3 · The check cell

Copy the whole next cell into your project notebook, **unchanged**, below your baseline run. The last output line carries two fingerprints: `cell` shows whether the cell was edited, `data` identifies your exact CSV files. Paste the full printed output into your appendix and upload it at the checkpoint.

In [26]:
# ===================== SSCM 2026 · GROUP PROJECT CHECK CELL · v1.0 =====================
# Copy this cell into your notebook UNCHANGED and run it after your baseline run.
# It needs two names defined earlier in your notebook:
#   DATA_DIR = the folder that holds your CSV files, e.g. DATA_DIR = "my_product_data"
#   baseline = your baseline results: one dictionary per seed (a list of 10), or one
#              dictionary with the means over the seeds. Keys (all numbers):
#     weeks_simulated      weeks in the run (52)
#     hours_simulated      clock hours of the run; the same clock as avg_hours_on_line
#     units_started        units that entered the line
#     units_finished       units that left the last station
#     key_components_used  key components taken from stock into the line
#     avg_units_on_line    time-average of units between start and finish
#     avg_hours_on_line    average hours from start to finish of a finished unit
#     cost_per_unit_eur    one-year total cost divided by units finished
# The grader runs the original of this cell on your files. Do not edit it.
import hashlib, math, os, datetime
import pandas as pd

CHECK_VERSION = "1.0"
REQUIRED = {
    "products.csv": ["product_id", "product_name", "frame_material", "mass_kg", "list_price_eur"],
    "bom.csv": ["product_id", "component_id", "component_name", "qty_per_unit", "mass_kg",
                "unit_cost_eur", "supplier_id", "added_at_workcenter"],
    "workcenters.csv": ["workcenter_id", "workcenter_name", "seq", "num_servers", "shift_hours",
                        "days_per_week", "availability", "process_time_cv", "fixed_cost_eur_week"],
    "routing.csv": ["product_id", "seq", "workcenter_id", "process_time_min", "setup_time_min"],
    "process_environment.csv": ["workcenter_id", "kw_running", "kw_idle", "waste_kg_per_unit",
                                "water_l_per_unit"],
    "suppliers.csv": ["supplier_id", "supplier_name", "country", "component_id", "unit_price_eur",
                      "promised_lead_days", "lead_time_mean_days", "lead_time_sd_days",
                      "capacity_per_month", "min_per_month", "kg_co2e_per_unit", "location_id"],
    "locations.csv": ["location_id", "name", "type", "country", "lat", "lon"],
    "network_lanes.csv": ["lane_id", "origin_id", "destination_id", "mode_id", "distance_km",
                          "transit_days_mean", "transit_days_sd"],
    "freight_rates.csv": ["lane_id", "mode_id", "rate_eur_per_shipment", "rate_eur_per_tkm",
                          "min_charge_eur"],
    "sources.csv": ["file", "row_id", "column", "value", "source_type", "source_or_calculation",
                    "sanity_check"],
}
MAY_BE_BLANK = {("bom.csv", "supplier_id")}           # parts not bought from a modelled supplier
MAY_BE_NEGATIVE = {("locations.csv", "lat"), ("locations.csv", "lon")}
KEYS = ["weeks_simulated", "hours_simulated", "units_started", "units_finished",
        "key_components_used", "avg_units_on_line", "avg_hours_on_line", "cost_per_unit_eur"]

def _cell_fingerprint():
    try:
        return hashlib.sha256(get_ipython().user_ns["_ih"][-1].strip().encode()).hexdigest()[:12]
    except Exception:
        return "n/a"

def _runs():
    b = globals().get("baseline")
    runs = b if isinstance(b, (list, tuple)) else [b]
    if b is None or not runs:
        raise ValueError("no `baseline` defined: run your baseline first")
    for i, r in enumerate(runs, 1):
        missing = [k for k in KEYS if k not in r]
        if missing:
            raise ValueError(f"baseline run {i} lacks {missing}")
        for k in KEYS:
            if not isinstance(r[k], (int, float)) or not math.isfinite(r[k]) or r[k] < 0:
                raise ValueError(f"baseline run {i}: {k} = {r[k]!r} is not a number ≥ 0")
        if r["units_finished"] <= 0 or r["hours_simulated"] <= 0 or r["weeks_simulated"] <= 0:
            raise ValueError(f"baseline run {i}: units_finished, hours and weeks must be > 0")
    return runs

def _verdict(name, ok, lines):
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    for line in lines:
        print("      " + line)
    return ok

data, results = {}, {}
folder = globals().get("DATA_DIR", ".")
h = hashlib.sha256()

# ---- T1 Format ----------------------------------------------------------------------
problems = []
for fname, cols in REQUIRED.items():
    path = os.path.join(folder, fname)
    if not os.path.exists(path):
        problems.append(f"{fname}: file missing in {folder!r}")
        continue
    h.update(open(path, "rb").read())
    df = pd.read_csv(path)
    df.columns = [str(c) for c in df.columns]
    data[fname] = df
    lacking = [c for c in cols if c not in df.columns]
    if lacking:
        problems.append(f"{fname}: column(s) missing or misspelt: {lacking}")
    if len(df) == 0:
        problems.append(f"{fname}: no rows")
    for c in [c for c in cols if c in df.columns]:
        blanks = df[c].isna() | (df[c].astype(str).str.strip() == "")
        if blanks.any() and (fname, c) not in MAY_BE_BLANK:
            problems.append(f"{fname}: {c} is empty in row(s) {list(df.index[blanks] + 2)}")
        num = pd.to_numeric(df[c], errors="coerce")
        if (num < 0).any() and (fname, c) not in MAY_BE_NEGATIVE:
            problems.append(f"{fname}: {c} is negative in row(s) {list(df.index[num < 0] + 2)}")
if "workcenters.csv" in data and "availability" in data["workcenters.csv"]:
    av = pd.to_numeric(data["workcenters.csv"]["availability"], errors="coerce")
    if ((av <= 0) | (av > 1) | av.isna()).any():
        problems.append("workcenters.csv: availability must be a share between 0 and 1")
if "routing.csv" in data and "process_time_min" in data["routing.csv"]:
    if (pd.to_numeric(data["routing.csv"]["process_time_min"], errors="coerce") <= 0).any():
        problems.append("routing.csv: every process_time_min must be above 0")
results["T1"] = _verdict("T1 Format", not problems,
                         problems[:12] + (["… and more"] if len(problems) > 12 else [])
                         or ["all 10 files present, columns exact, no blanks or negatives"])

# ---- baseline numbers for T2–T5 -------------------------------------------------------
try:
    runs = _runs()
    base_error = None
except Exception as e:
    runs, base_error = [], str(e)
if base_error:
    for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
        results[t[:2]] = _verdict(t, False, [base_error])
else:
    try:
        prod = data["products.csv"].iloc[0]
        pid = prod["product_id"]
        wc = data["workcenters.csv"].set_index("workcenter_id")
        rt = data["routing.csv"]
        rt = rt[rt["product_id"] == pid]
        minutes = rt.groupby("workcenter_id")["process_time_min"].sum()
        unknown = [w for w in minutes.index if w not in wc.index]
        avail_min_week = (wc["num_servers"] * wc["shift_hours"] * 60 * wc["days_per_week"]
                          * wc["availability"])
        # ---- T2 Capacity: load offered by the units started, per station
        lines, ok = [], not unknown and len(minutes) > 0
        if unknown:
            lines.append(f"routing.csv names stations not in workcenters.csv: {unknown}")
        worst = {}
        for r in runs:
            for w, m in minutes.items():
                if w in wc.index:
                    share = r["units_started"] * m / (avail_min_week[w] * r["weeks_simulated"])
                    worst[w] = max(worst.get(w, 0), share)
        for w, share in worst.items():
            flag = "  <- 100 % or more" if share >= 1 else ""
            lines.append(f"{w} {wc.loc[w, 'workcenter_name']}: busy {share:.1%} of available time{flag}")
            ok = ok and share < 1
        results["T2"] = _verdict("T2 Capacity", ok, lines)
        # ---- T3 Little's Law: L = λ × W, within 10 %
        lines, ok = [], True
        for i, r in enumerate(runs, 1):
            lam = r["units_finished"] / r["hours_simulated"]
            lw = lam * r["avg_hours_on_line"]
            gap = abs(r["avg_units_on_line"] - lw) / lw if lw > 0 else float("inf")
            ok = ok and gap <= 0.10
            if (gap > 0.10 and len(lines) < 3) or i == 1:
                lines.append(f"run {i}: L = {r['avg_units_on_line']:.2f}  vs  λ×W = {lam:.3f}/h × "
                             f"{r['avg_hours_on_line']:.2f} h = {lw:.2f}  (gap {gap:.1%})")
        results["T3"] = _verdict("T3 Little's Law", ok, lines + [f"{len(runs)} run(s) checked"])
        # ---- T4 Flow balance
        bom = data["bom.csv"]
        bom = bom[bom["product_id"] == pid]
        keys = bom[bom["component_id"].isin(data["suppliers.csv"]["component_id"])]
        lines, ok = [], len(keys) == 1
        if len(keys) != 1:
            lines.append(f"expected one key component (in bom.csv and suppliers.csv), found {len(keys)}")
        else:
            q = float(keys["qty_per_unit"].iloc[0])
            n_opt = int((data["suppliers.csv"]["component_id"] == keys["component_id"].iloc[0]).sum())
            if n_opt < 2:
                lines.append(f"note: only {n_opt} sourcing option in suppliers.csv (the brief asks for two)")
            for i, r in enumerate(runs, 1):
                s, f, u = r["units_started"], r["units_finished"], r["key_components_used"]
                good = f <= s + 0.5 and f * q - 0.5 <= u <= s * q + 0.5
                ok = ok and good
                if not good or i == 1:
                    lines.append(f"run {i}: started {s:,.0f}, finished {f:,.0f}, key components used "
                                 f"{u:,.0f} (allowed {f*q:,.0f} to {s*q:,.0f})")
        results["T4"] = _verdict("T4 Flow balance", ok, lines)
        # ---- T5 Margin
        price = float(prod["list_price_eur"])
        material = float((bom["qty_per_unit"] * bom["unit_cost_eur"]).sum())
        costs = [r["cost_per_unit_eur"] for r in runs]
        lines = [f"cost per unit {min(costs):,.2f} to {max(costs):,.2f} EUR vs list price {price:,.2f} EUR"]
        if min(costs) < material:
            lines.append(f"warning: below the material cost alone ({material:,.2f} EUR per unit)")
        results["T5"] = _verdict("T5 Margin", max(costs) < price, lines)
    except Exception as e:
        for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
            if t[:2] not in results:
                results[t[:2]] = _verdict(t, False, [f"could not run: {e}"])

print("-" * 78)
print(f"{sum(results.values())} of 5 tests passed  ·  {'ALL PASS' if all(results.values()) and len(results) == 5 else 'NOT YET'}")
print(f"check cell v{CHECK_VERSION} · cell {_cell_fingerprint()} · data {h.hexdigest()[:12]} · "
      f"{datetime.datetime.now():%Y-%m-%d %H:%M}")
# ======================= end of check cell · do not edit above ==========================

PASS  T1 Format
      all 10 files present, columns exact, no blanks or negatives
PASS  T2 Capacity
      WC1 SMT: busy 64.8% of available time
      WC2 Board test: busy 32.4% of available time
      WC3 Assembly: busy 51.5% of available time
      WC4 Final test: busy 48.6% of available time
      WC5 Packaging: busy 15.4% of available time
PASS  T3 Little's Law
      run 1: L = 2.84  vs  λ×W = 17.499/h × 0.16 h = 2.84  (gap 0.0%)
      10 run(s) checked
PASS  T4 Flow balance
      run 1: started 36,400, finished 36,397, key components used 36,400 (allowed 36,397 to 36,400)
PASS  T5 Margin
      cost per unit 397.44 to 397.47 EUR vs list price 755.46 EUR
------------------------------------------------------------------------------
5 of 5 tests passed  ·  ALL PASS
check cell v1.0 · cell 3ad30383c119 · data cfe836d3781c · 2026-11-11 20:09


In [29]:
# 11. Create the combined interactive supply chain dashboard

import json
import numpy as np
import pandas as pd
import heapq
import os
from IPython.display import HTML, display

# Ensure DATA_DIR and basic data imports exist inside this cell context
DATA_DIR = globals().get("DATA_DIR", "/content/team3_data")
REPO_URL = "https://github.com/TH-Koln-Bartnik/sscm-2026-student"  # course website copy: run the data cell at the top first

# Automatically clone the repository if it's missing to make the cell completely self-contained
if not os.path.exists(DATA_DIR):
    print(f"DATA_DIR not found at {DATA_DIR}. Cloning repository from {REPO_URL}...")
    import subprocess
    subprocess.run(["git", "clone", "-q", REPO_URL, DATA_DIR])

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found at {DATA_DIR} even after cloning attempt. Please verify network connection or REPO_URL.")

# Load all required DataFrames locally to prevent NameErrors after a kernel reset
suppliers = pd.read_csv(os.path.join(DATA_DIR, "suppliers.csv"))
network_lanes = pd.read_csv(os.path.join(DATA_DIR, "network_lanes.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
bom = pd.read_csv(os.path.join(DATA_DIR, "bom.csv"))
workcenters = pd.read_csv(os.path.join(DATA_DIR, "workcenters.csv"))
routing = pd.read_csv(os.path.join(DATA_DIR, "routing.csv"))
freight_rates = pd.read_csv(os.path.join(DATA_DIR, "freight_rates.csv"))
process_environment = pd.read_csv(os.path.join(DATA_DIR, "process_environment.csv"))

PRODUCT_ID = "PH-FLAG"
product_bom = bom[bom["product_id"] == PRODUCT_ID].copy()
product_routing = routing[routing["product_id"] == PRODUCT_ID].sort_values("seq").copy()

# Resolve needed assumptions locally
ELECTRICITY_CO2_FACTOR = 0.344  # kg CO2 per kWh
SHIPMENT_SIZE = 700
ANNUAL_INVENTORY_HOLDING_RATE = 0.20

supplier_s1 = suppliers.loc[suppliers["supplier_id"] == "S1"].iloc[0]
supplier_s3 = suppliers.loc[suppliers["supplier_id"] == "S3"].iloc[0]
memory_row = product_bom.loc[product_bom["component_id"] == "mem"].iloc[0]
memory_qty = float(memory_row["qty_per_unit"])
memory_mass_kg = float(memory_row["mass_kg"]) * memory_qty
other_material_cost = float(product_bom.loc[product_bom["component_id"] != "mem", "unit_cost_eur"].sum())

def calculate_lane_freight(lane_id, shipment_mass_kg):
    lane = network_lanes.loc[network_lanes["lane_id"] == lane_id].iloc[0]
    rate = freight_rates.loc[freight_rates["lane_id"] == lane_id].iloc[0]
    tonne_km = float(lane["distance_km"]) * shipment_mass_kg / 1000
    variable_cost = float(rate["rate_eur_per_tkm"]) * tonne_km
    return max(float(rate["min_charge_eur"]), float(rate["rate_eur_per_shipment"]) + variable_cost)

s1_freight_per_phone = (calculate_lane_freight("L1-SEA", memory_mass_kg * SHIPMENT_SIZE) + calculate_lane_freight("L3-TRUCK", memory_mass_kg * SHIPMENT_SIZE)) / SHIPMENT_SIZE
s3_freight_per_phone = (calculate_lane_freight("L4-TRUCK", memory_mass_kg * SHIPMENT_SIZE)) / SHIPMENT_SIZE

energy_kwh_per_phone = 0.0
for _, station in product_routing.iterrows():
    wc_id = station["workcenter_id"]
    environment = process_environment.loc[process_environment["workcenter_id"] == wc_id].iloc[0]
    process_hours = float(station["process_time_min"]) / 60
    availability = float(workcenters.loc[workcenters["workcenter_id"] == wc_id, "availability"].iloc[0])
    energy_kwh_per_phone += float(environment["kw_running"]) * process_hours / availability

if 'SCENARIOS' not in globals():
    SCENARIOS = {
        "Baseline": {
            "description": "Normal operation without supply chain disruption.",
            "disruption_weeks": 0,
            "extra_lead_time_days": 0,
            "freight_cost_increase": 0.0,
            "dual_sourcing": False,
            "poland_share": 0.0
        },
        "Red Sea Crisis": {
            "description": "Asian shipments are rerouted around the Cape of Good Hope.",
            "disruption_weeks": 12,
            "extra_lead_time_days": 10,
            "freight_cost_increase": 0.20,
            "dual_sourcing": False,
            "poland_share": 0.0
        },
        "Red Sea Crisis + Dual Sourcing": {
            "description": "A second supplier in Poland is activated.",
            "disruption_weeks": 12,
            "extra_lead_time_days": 10,
            "freight_cost_increase": 0.20,
            "dual_sourcing": True,
            "poland_share": 0.50
        }
    }

if 'WORKING_HOURS_PER_WEEK' not in globals():
    WORKING_HOURS_PER_WEEK = 40
if 'WEEKLY_MEMORY_DEMAND' not in globals():
    WEEKLY_MEMORY_DEMAND = 700
if 'INITIAL_MEMORY_STOCK' not in globals():
    INITIAL_MEMORY_STOCK = 5 * WEEKLY_MEMORY_DEMAND
if 'WEEKS' not in globals():
    WEEKS = 52
if 'HOURS_SIMULATED' not in globals():
    HOURS_SIMULATED = 52 * 5 * 8

# --------------------------------------------------
# 1. Advanced Real-World Material Supply Simulation
# --------------------------------------------------

def simulate_real_material_supply(scenario_name, seed=1):
    rng = np.random.default_rng(seed)
    scenario = SCENARIOS[scenario_name]

    stock = INITIAL_MEMORY_STOCK
    deliveries = {}  # week -> arrived qty

    released_units = []
    stopped_hours = 0.0
    weekly_results = []

    s1 = suppliers.loc[suppliers["supplier_id"] == "S1"].iloc[0]
    s3 = suppliers.loc[suppliers["supplier_id"] == "S3"].iloc[0]

    l1_sea = network_lanes.loc[network_lanes["lane_id"] == "L1-SEA"].iloc[0]
    l3_truck = network_lanes.loc[network_lanes["lane_id"] == "L3-TRUCK"].iloc[0]
    l4_truck = network_lanes.loc[network_lanes["lane_id"] == "L4-TRUCK"].iloc[0]

    for week in range(WEEKS):
        stock += deliveries.get(week, 0)

        if scenario["dual_sourcing"]:
            s1_share = 1.0 - scenario["poland_share"]
            s3_share = scenario["poland_share"]
        else:
            s1_share = 1.0
            s3_share = 0.0

        s1_order = WEEKLY_MEMORY_DEMAND * s1_share
        s3_order = WEEKLY_MEMORY_DEMAND * s3_share

        if s1_order > 0:
            prod_lead = rng.normal(float(s1["lead_time_mean_days"]), float(s1["lead_time_sd_days"]))
            prod_lead = max(1.0, prod_lead)

            if week < scenario["disruption_weeks"]:
                prod_lead += scenario["extra_lead_time_days"]

            t1 = rng.normal(float(l1_sea["transit_days_mean"]), float(l1_sea["transit_days_sd"]))
            t1 = max(1.0, t1)

            t3 = rng.normal(float(l3_truck["transit_days_mean"]), float(l3_truck["transit_days_sd"]))
            t3 = max(0.5, t3)

            total_lead_days = prod_lead + t1 + t3
            arrival_week = week + int(np.ceil(total_lead_days / 7))

            if arrival_week < WEEKS:
                deliveries[arrival_week] = deliveries.get(arrival_week, 0) + s1_order

        if s3_order > 0:
            prod_lead = rng.normal(float(s3["lead_time_mean_days"]), float(s3["lead_time_sd_days"]))
            prod_lead = max(1.0, prod_lead)

            t4 = rng.normal(float(l4_truck["transit_days_mean"]), float(l4_truck["transit_days_sd"]))
            t4 = max(0.5, t4)

            total_lead_days = prod_lead + t4
            arrival_week = week + int(np.ceil(total_lead_days / 7))

            if arrival_week < WEEKS:
                deliveries[arrival_week] = deliveries.get(arrival_week, 0) + s3_order

        available_units = min(WEEKLY_MEMORY_DEMAND, int(stock))
        stock -= available_units

        missing_units = WEEKLY_MEMORY_DEMAND - available_units
        weekly_stopped_hours = (missing_units / WEEKLY_MEMORY_DEMAND) * WORKING_HOURS_PER_WEEK
        stopped_hours += weekly_stopped_hours

        if available_units > 0:
            week_starts = (
                week * WORKING_HOURS_PER_WEEK
                + np.linspace(0, WORKING_HOURS_PER_WEEK, available_units, endpoint=False)
            )
            released_units.extend(week_starts.tolist())

        weekly_results.append({
            "week": week + 1,
            "memory_stock": stock,
            "units_released": available_units,
            "missing_units": missing_units,
            "stopped_hours": weekly_stopped_hours
        })

    return {
        "starts": np.array(released_units, dtype=float),
        "stopped_hours": stopped_hours,
        "weekly_results": pd.DataFrame(weekly_results)
    }

# --------------------------------------------------
# 2. Dynamic Scenario Evaluation for Dashboard
# --------------------------------------------------

scenario_keys = {
    "Baseline": "baseline",
    "Red Sea Crisis": "disruption",
    "Red Sea Crisis + Dual Sourcing": "countermeasure"
}

animation_data = {
    "product": "Samsung Galaxy S25 Ultra 256GB",
    "unit": "smartphones",
    "component": "Memory chips",
    "stations": ["SMT", "Board test", "Assembly", "Final test", "Packaging"],
    "scenarios": {}
}

def line_averages(start_h, finish_h, hours_simulated):
    start_h = np.asarray(start_h, dtype=float)
    finish_h = np.asarray(finish_h, dtype=float)
    completed = finish_h <= hours_simulated
    clipped_finish = np.minimum(finish_h, hours_simulated)
    avg_units_on_line = np.maximum(0, clipped_finish - start_h).sum() / hours_simulated
    avg_hours_on_line = (finish_h[completed] - start_h[completed]).mean() if completed.any() else 0.0
    return float(avg_units_on_line), float(avg_hours_on_line)

for scenario_name, scenario_key in scenario_keys.items():
    supply = simulate_real_material_supply(scenario_name, seed=1)
    weekly = supply["weekly_results"]

    starts = supply["starts"]
    stopped_hours = supply["stopped_hours"]
    rng = np.random.default_rng(1)

    route = product_routing.merge(
        workcenters[["workcenter_id", "num_servers", "availability", "process_time_cv"]],
        on="workcenter_id",
        how="left",
        validate="one_to_one"
    )

    station_arrivals = starts.copy()
    for _, station in route.iterrows():
        servers = int(station["num_servers"])
        availability = float(station["availability"])
        cv = float(station["process_time_cv"])
        mean_process_hours = float(station["process_time_min"]) / 60
        effective_mean = mean_process_hours / availability

        sigma_squared = np.log(1 + cv**2)
        sigma = np.sqrt(sigma_squared)
        mu = np.log(effective_mean) - sigma_squared / 2
        service_times = rng.lognormal(mu, sigma, len(station_arrivals))

        server_free_times = [0.0] * servers
        heapq.heapify(server_free_times)
        departures = np.empty(len(station_arrivals))

        for i, arrival in enumerate(station_arrivals):
            earliest_server = heapq.heappop(server_free_times)
            service_start = max(float(arrival), earliest_server)
            departure = service_start + float(service_times[i])
            departures[i] = departure
            heapq.heappush(server_free_times, departure)

        station_arrivals = departures

    units_finished = int(np.sum(station_arrivals <= HOURS_SIMULATED))
    avg_units, avg_hours = line_averages(starts, station_arrivals, HOURS_SIMULATED)

    scenario_config = SCENARIOS[scenario_name]
    poland_share = scenario_config["poland_share"]
    asia_share = 1.0 - poland_share

    memory_cost = memory_qty * (
        asia_share * float(supplier_s1["unit_price_eur"])
        + poland_share * float(supplier_s3["unit_price_eur"])
    )
    material_cost = other_material_cost + memory_cost

    freight_cost = (
        asia_share * s1_freight_per_phone
        + poland_share * s3_freight_per_phone
    )
    if scenario_config["disruption_weeks"] > 0:
        freight_cost = (
            asia_share * s1_freight_per_phone * (1.0 + scenario_config["freight_cost_increase"])
            + poland_share * s3_freight_per_phone
        )

    annual_fixed_cost = float(workcenters["fixed_cost_eur_week"].sum()) * WEEKS
    production_cost = annual_fixed_cost / units_finished if units_finished > 0 else 0

    average_memory_stock = float(weekly["memory_stock"].mean())
    average_memory_price = (
        asia_share * float(supplier_s1["unit_price_eur"])
        + poland_share * float(supplier_s3["unit_price_eur"])
    )
    average_inventory_value = average_memory_stock * average_memory_price
    annual_holding_cost = average_inventory_value * ANNUAL_INVENTORY_HOLDING_RATE
    holding_cost_per_phone = annual_holding_cost / units_finished if units_finished > 0 else 0

    total_cost = material_cost + freight_cost + production_cost + holding_cost_per_phone

    supplier_emissions = memory_qty * (
        asia_share * float(supplier_s1["kg_co2e_per_unit"])
        + poland_share * float(supplier_s3["kg_co2e_per_unit"])
    )
    electricity_emissions = energy_kwh_per_phone * ELECTRICITY_CO2_FACTOR
    total_emissions = supplier_emissions + electricity_emissions

    if scenario_config["dual_sourcing"]:
        suppliers_text = "S1 MemCore (50%) + S3 MemEU (50%)"
    else:
        suppliers_text = "S1 MemCore (100%)"

    animation_data["scenarios"][scenario_key] = {
        "label": scenario_name,
        "suppliers": suppliers_text,
        "units_started": len(starts),
        "units_finished": units_finished,
        "units_per_week": float(units_finished) / 52,
        "hours_stopped": float(stopped_hours),
        "avg_units_on_line": float(avg_units),
        "cost": float(total_cost),
        "co2": float(total_emissions),
        "weeks": weekly.to_dict(orient="records")
    }

json_data = json.dumps(animation_data, ensure_ascii=False)
# Export simulation data as result.json
with open("result.json", "w", encoding="utf-8") as file:
    json.dump(animation_data, file, ensure_ascii=False, indent=4)

print("result.json successfully created.")

# --------------------------------------------------
# 3. Full Complete HTML Layout Template & JavaScript
# --------------------------------------------------

html_content = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Smartphone Supply Chain Dashboard</title>
<style>
:root {
    --bg-dark: #0f172a;
    --card-bg: #1e293b;
    --accent-blue: #38bdf8;
    --accent-green: #34d399;
    --accent-red: #f87171;
    --accent-orange: #fb923c;
    --text-main: #f8fafc;
    --text-muted: #94a3b8;
    --border-color: #334155;
}
* { box-sizing: border-box; }
body {
    background: var(--bg-dark);
    color: var(--text-main);
    font-family: 'Segoe UI', Arial, sans-serif;
    margin: 0;
    padding: 20px;
}
.container {
    max-width: 1200px;
    margin: auto;
}
.header {
    border-bottom: 2px solid var(--border-color);
    padding-bottom: 15px;
    margin-bottom: 25px;
}
.header h1 {
    margin: 0;
    font-size: 25px;
    color: var(--accent-blue);
}
.header-meta {
    color: var(--text-muted);
    font-size: 13px;
    margin-top: 6px;
}
.controls-card {
    background: var(--card-bg);
    border: 1px solid var(--border-color);
    border-radius: 12px;
    padding: 16px;
    margin-bottom: 22px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 15px;
    flex-wrap: wrap;
}
.btn {
    background: #2563eb;
    color: white;
    border: none;
    padding: 10px 16px;
    border-radius: 7px;
    font-weight: bold;
    cursor: pointer;
}
.btn:hover { background: #1d4ed8; }
.slider-container {
    flex: 1;
    min-width: 220px;
    display: flex;
    align-items: center;
    gap: 12px;
    background: rgba(15,23,42,.4);
    padding: 10px;
    border-radius: 8px;
}
input[type="range"] {
    flex: 1;
    accent-color: var(--accent-blue);
}
.scenario-tabs {
    display: flex;
    gap: 5px;
    flex-wrap: wrap;
}
.tab {
    padding: 9px 12px;
    border-radius: 6px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    border: none;
    background: transparent;
    color: var(--text-muted);
}
.tab.active {
    background: #0369a1;
    color: white;
}
.tab[data-scen="disruption"].active {
    background: #991b1b;
}
.tab[data-scen="countermeasure"].active {
    background: #065f46;
}
.metrics-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
    margin-bottom: 22px;
}
.metric-card, .panel {
    background: var(--card-bg);
    border: 1px solid var(--border-color);
    border-radius: 12px;
    padding: 17px;
}
.metric-label {
    color: var(--text-muted);
    font-size: 11px;
    text-transform: uppercase;
    margin-bottom: 8px;
}
.metric-value {
    font-size: 22px;
    font-weight: bold;
}
.pipeline-card {
    background: var(--card-bg);
    border: 1px solid var(--border-color);
    border-radius: 12px;
    padding: 28px 18px;
    margin-bottom: 22px;
    overflow-x: auto;
}
.pipeline-track {
    display: flex;
    align-items: center;
    justify-content: space-between;
    min-width: 900px;
}
.node {
    background: var(--bg-dark);
    border: 2px solid var(--border-color);
    border-radius: 10px;
    padding: 12px 8px;
    width: 140px;
    text-align: center;
    transition: .3s;
}
.node-title {
    font-size: 11px;
    color: var(--text-muted);
    text-transform: uppercase;
}
.node-value {
    font-size: 14px;
    font-weight: bold;
    margin-top: 7px;
    color: var(--accent-blue);
}
.node.active-warn {
    border-color: var(--accent-red);
    box-shadow: 0 0 12px rgba(248,113,113,.4);
}
.node.active-good {
    border-color: var(--accent-green);
}
.connection-lane {
    flex: 1;
    height: 4px;
    min-width: 15px;
    background: var(--border-color);
    margin: 0 -5px;
}
.connection-lane.active-flow {
    background: linear-gradient(to right, var(--accent-blue), var(--accent-green));
}
.info-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 18px;
    margin-bottom: 22px;
}
.panel h3 {
    color: var(--accent-blue);
    font-size: 14px;
    margin-top: 0;
}
.log-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}
.log-table td {
    padding: 9px 0;
    border-bottom: 1px solid rgba(51,65,85,.6);
}
.log-val {
    text-align: right;
    font-weight: bold;
}
.chart-card {
    background: var(--card-bg);
    border: 1px solid var(--border-color);
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 22px;
}
.chart-title {
    color: var(--accent-blue);
    font-size: 14px;
    font-weight: bold;
    margin-bottom: 12px;
}
.chart-svg {
    width: 100%;
    height: 230px;
}
.small {
    color: var(--text-muted);
    font-size: 12px;
}
@media(max-width:750px) {
    .metrics-grid { grid-template-columns: repeat(2,1fr); }
    .info-grid { grid-template-columns: 1fr; }
}
</style>
</head>
<body>
<div class="container">
    <div class="header">
        <h1 id="product-title">Supply Chain Dashboard</h1>
        <div class="header-meta">52-week smartphone production simulation</div>
    </div>
    <div class="controls-card">
        <button class="btn" id="play-btn">▶ Play</button>
        <div class="slider-container">
            <span>Week 1</span>
            <input type="range" id="time-slider" min="0" max="51" value="0">
            <span id="week-label">Week 1</span>
        </div>
        <div class="scenario-tabs" id="scenario-selector"></div>
    </div>
    <div class="metrics-grid">
        <div class="metric-card">
            <div class="metric-label">Finished smartphones</div>
            <div class="metric-value" id="metric-finished">-</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Cost per unit</div>
            <div class="metric-value" id="metric-cost">-</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">CO2 footprint per unit</div>
            <div class="metric-value" id="metric-co2">-</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Production stopped</div>
            <div class="metric-value" id="metric-stopped">-</div>
        </div>
    </div>
    <div class="pipeline-card">
        <div class="pipeline-track" id="pipeline-track"></div>
    </div>
    <div class="info-grid">
        <div class="panel">
            <h3>Supply Chain Status</h3>
            <table class="log-table">
                <tr>
                    <td>Units released this week</td>
                    <td class="log-val" id="w-out">-</td>
                </tr>
                <tr>
                    <td>Missing memory chips</td>
                    <td class="log-val" id="w-missing">-</td>
                </tr>
                <tr>
                    <td>Memory stock</td>
                    <td class="log-val" id="w-stock">-</td>
                </tr>
                <tr>
                    <td>Stopped hours this week</td>
                    <td class="log-val" id="w-stopped">-</td>
                </tr>
                <tr>
                    <td>Average units on production line</td>
                    <td class="log-val" id="w-wip">-</td>
                </tr>
            </table>
        </div>
        <div class="panel">
            <h3>Disruption Details</h3>
            <p id="disruption-status" class="small"></p>
            <p id="supplier-info" class="small"></p>
        </div>
    </div>
    <div class="chart-card">
        <div class="chart-title">Memory Chip Stock Development</div>
        <svg id="stock-chart" class="chart-svg" viewBox="0 0 900 230" preserveAspectRatio="none"></svg>
        <div class="small">Actual weekly stock values from our material supply simulation.</div>
    </div>
    <div class="chart-card">
        <div class="chart-title">Weekly Material Supply</div>
        <svg id="supply-chart" class="chart-svg" viewBox="0 0 900 230" preserveAspectRatio="none"></svg>
        <div class="small">Blue: units released to production. Red: missing units.</div>
    </div>
</div>
<script>
const data = __DATA__;
let currentWeek = 0;
let currentScenario = "baseline";
let isPlaying = false;
let playInterval = null;

const scenarioSelector = document.getElementById("scenario-selector");
document.getElementById("product-title").innerText = data.product + " - Supply Chain";

Object.keys(data.scenarios).forEach((key, index) => {
    const button = document.createElement("button");
    button.className = "tab" + (index === 0 ? " active" : "");
    button.dataset.scen = key;
    button.innerText = data.scenarios[key].label;
    button.onclick = () => selectScenario(key);
    scenarioSelector.appendChild(button);
});

function selectScenario(key) {
    currentScenario = key;
    currentWeek = 0;
    document.querySelectorAll(".tab").forEach(button => {
        button.classList.toggle("active", button.dataset.scen === key);
    });
    updateDashboard();
}

function stopPlayback() {
    if (playInterval) clearInterval(playInterval);
    playInterval = null;
    isPlaying = false;
    document.getElementById("play-btn").innerText = "▶ Play";
}

function updateDashboard() {
    const scenario = data.scenarios[currentScenario];
    const week = scenario.weeks[currentWeek];

    document.getElementById("metric-finished").innerText = Math.round(scenario.units_per_week).toLocaleString("en-US") + " / week";
    document.getElementById("metric-cost").innerText = scenario.cost.toFixed(2) + " €";
    document.getElementById("metric-co2").innerText = scenario.co2.toFixed(4) + " kg";
    document.getElementById("metric-stopped").innerText = scenario.hours_stopped.toFixed(1) + " h";
    document.getElementById("time-slider").value = currentWeek;
    document.getElementById("week-label").innerText = "Week " + (currentWeek + 1);
    document.getElementById("w-out").innerText = week.units_released.toLocaleString("en-US");
    document.getElementById("w-missing").innerText = week.missing_units.toLocaleString("en-US");
    document.getElementById("w-stock").innerText = Math.round(week.memory_stock).toLocaleString("en-US");
    document.getElementById("w-stopped").innerText = week.stopped_hours.toFixed(1) + " h";
    document.getElementById("w-wip").innerText = scenario.avg_units_on_line.toFixed(2) + " (annual average)";
    document.getElementById("supplier-info").innerText = "Active sourcing: " + scenario.suppliers;

    const status = document.getElementById("disruption-status");
    if (week.missing_units > 0) {
        status.innerHTML = '<b style="color:var(--accent-red)">WARNING:</b> ' + week.missing_units + ' memory chips are missing. Production is partially stopped.';
    } else {
        status.innerHTML = '<b style="color:var(--accent-green)">NORMAL:</b> Sufficient memory chips are available.';
    }
    drawPipeline(scenario, week);
    drawStockChart(scenario);
    drawSupplyChart(scenario);
}

function drawPipeline(scenario, week) {
    const track = document.getElementById("pipeline-track");
    track.innerHTML = "";

    const supplierNode = document.createElement("div");
    supplierNode.className = "node " + (week.missing_units > 0 ? "active-warn" : "active-good");
    supplierNode.innerHTML = `<div class="node-title">Memory suppliers</div><div class="node-value">${week.missing_units > 0 ? "SHORTAGE" : "ONLINE"}</div><div class="small">${scenario.suppliers}</div>`;
    track.appendChild(supplierNode);

    const lane = document.createElement("div");
    lane.className = "connection-lane active-flow";
    track.appendChild(lane);

    const plant = document.createElement("div");
    plant.className = "node";
    plant.innerHTML = `<div class="node-title">Transport</div><div class="node-value">Supply</div><div class="small">Memory chips</div>`;
    track.appendChild(plant);

    const lane2 = document.createElement("div");
    lane2.className = "connection-lane active-flow";
    track.appendChild(lane2);

    data.stations.forEach((station, index) => {
        const node = document.createElement("div");
        node.className = "node " + (week.missing_units > 0 ? "active-warn" : "");
        node.innerHTML = `<div class="node-title">${station}</div><div class="node-value">${week.missing_units > 0 ? "RESTRICTED" : "NOMINAL"}</div>`;
        track.appendChild(node);
        if (index < data.stations.length - 1) {
            const nextLane = document.createElement("div");
            nextLane.className = "connection-lane active-flow";
            track.appendChild(nextLane);
        }
    });
}

function drawStockChart(scenario) {
    const values = scenario.weeks.map(w => w.memory_stock);
    const width = 900; const height = 210; const left = 45; const right = 15; const top = 15; const bottom = 35;
    const max = Math.max(1, ...values) * 1.1; const min = Math.min(0, ...values);
    const x = i => left + i * (width-left-right) / 51;
    const y = v => height-bottom - (v-min) / (max-min || 1) * (height-top-bottom);
    const points = values.map((v,i) => `${x(i)},${y(v)}`).join(" ");
    const grid = [0, .25, .5, .75, 1].map(f => {
        const gy = top + f * (height-top-bottom);
        return `<line x1="${left}" y1="${gy}" x2="${width-right}" y2="${gy}" stroke="#334155" stroke-dasharray="4,4"/>`;
    }).join("");
    const markerX = x(currentWeek);
    document.getElementById("stock-chart").innerHTML = `
        ${grid}
        <polyline points="${points}" fill="none" stroke="#34d399" stroke-width="3" stroke-linejoin="round" stroke-linecap="round"/>
        <line x1="${markerX}" y1="${top}" x2="${markerX}" y2="${height-bottom}" stroke="#facc15" stroke-width="2" stroke-dasharray="5,5"/>
        <circle cx="${markerX}" cy="${y(values[currentWeek])}" r="6" fill="#facc15"/>
        <text x="${left}" y="${height-8}" fill="#94a3b8" font-size="11">Week 1</text>
        <text x="${width-right}" y="${height-8}" text-anchor="end" fill="#94a3b8" font-size="11">Week 52</text>
    `;
}

function drawSupplyChart(scenario) {
    const weeks = scenario.weeks;
    const width = 900; const height = 210; const left = 35; const bottom = 30; const top = 10;
    const max = Math.max(1, ...weeks.map(w => Math.max(w.units_released, w.missing_units)));
    const barWidth = 10; const gap = (width-left-10) / 52;
    let bars = "";
    weeks.forEach((w, i) => {
        const x = left + i * gap;
        const h1 = w.units_released / max * (height-top-bottom);
        const h2 = w.missing_units / max * (height-top-bottom);
        bars += `
            <rect x="${x}" y="${height-bottom-h1}" width="${barWidth}" height="${h1}" fill="#38bdf8" rx="2"/>
            <rect x="${x+barWidth+1}" y="${height-bottom-h2}" width="${barWidth}" height="${h2}" fill="#f87171" rx="2"/>
        `;
    });
    const markerX = left + currentWeek * gap + barWidth;
    document.getElementById("supply-chart").innerHTML = `
        <line x1="${left}" y1="${height-bottom}" x2="${width}" y2="${height-bottom}" stroke="#64748b"/>
        ${bars}
        <line x1="${markerX}" y1="${top}" x2="${markerX}" y2="${height-bottom}" stroke="#facc15" stroke-width="2" stroke-dasharray="5,5"/>
    `;
}

document.getElementById("time-slider").oninput = function() {
    currentWeek = Number(this.value);
    updateDashboard();
};

document.getElementById("play-btn").onclick = function() {
    if (isPlaying) {
        stopPlayback();
        return;
    }
    if (currentWeek >= 51) currentWeek = 0;
    isPlaying = true;
    this.innerText = "⏸ Pause";
    playInterval = setInterval(() => {
        if (currentWeek >= 51) {
            stopPlayback();
            return;
        }
        currentWeek++;
        updateDashboard();
    }, 450);
};

updateDashboard();
</script>
</body>
</html>
"""

html_filled = html_content.replace("const data = __DATA__;", f"const data = {json_data};")

with open("animation_simulation.html", "w", encoding="utf-8") as file:
    file.write(html_filled)

display(HTML(html_filled))
print("Dashboard successfully updated and rendered under real-world stochastic conditions.")


result.json successfully created.


Units released this week,-
Missing memory chips,-
Memory stock,-
Stopped hours this week,-
Average units on production line,-


Dashboard successfully updated and rendered under real-world stochastic conditions.


## What to do with a FAIL

| FAIL in | Typical cause | Where to look |
|---|---|---|
| T1 Format | a column name with a typo, an empty cell, a minus sign, a file not uploaded | the line names the file, the column and the row |
| T2 Capacity | the line starts more units than its slowest station can handle | raise servers, shift hours or availability at that station, or start fewer units |
| T3 Little's Law | two clocks mixed (working hours vs calendar hours), or units counted in some places and not in others | measure both averages over the same units and the same clock |
| T4 Flow balance | units created without a key component, or components consumed twice | count the key component where it enters the line |
| T5 Margin | a cost counted per week instead of per year, or a price too low | recompute the total cost line by line |

_Prepared with Claude (Anthropic) as a coding assistant; checked and released by Professor Bartnik._